# Lesson 3: Can a large language model master Wordle?

Start by load dependencies and setting up the Predibase client, which you'll use to call both base and finetuned models:

In [ ]:
import os

from dotenv import load_dotenv
from openai import OpenAI

# Initialize client to query Qwen2.5 7B Instruct on Predibase using the OpenAI client.

_ = load_dotenv()

#client = OpenAI(
#    base_url=os.environ["PREDIBASE_MODEL_QWEN_URL"], # Qwen 2.5 7B Instruct
#    api_key=os.environ["PREDIBASE_API_KEY"],
#)

In [ ]:
from typing import Generator, Optional
from transformers import AutoTokenizer, AutoModelForCausalLM

class LLMHelper:
    """
    Helper class to interface with either Hugging Face LLMs (e.g., Qwen2.5-7B-Instruct) or Predibase API.
    """

    def __init__(self, use_hf: bool = True, hf_model_name: str = "Qwen/Qwen2.5-3B-Instruct") -> None:
        """
        Initialize the LLMHelper.

        Parameters
        ----------
        use_hf : bool, optional
            Whether to use Hugging Face model for inference (default is False).
        hf_model_name : str, optional
            Hugging Face model name to load (default is "Qwen/Qwen2.5-7B-Instruct").
        """
        self.use_hf = use_hf
        if self.use_hf:
            self.tokenizer = AutoTokenizer.from_pretrained(hf_model_name, trust_remote_code=True)
            self.model = AutoModelForCausalLM.from_pretrained(hf_model_name, trust_remote_code=True).to("cuda" if torch.cuda.is_available() else "cpu")

    def hf_generate(
        self,
        prompt: str,
        max_new_tokens: int = 256,
        temperature: float = 0.7,
        top_p: float = 0.95
    ) -> str:
        """
        Generate a completion using a Hugging Face model.

        Parameters
        ----------
        prompt : str
            Input prompt to the model.
        max_new_tokens : int, optional
            Maximum number of new tokens to generate (default is 256).
        temperature : float, optional
            Sampling temperature (default is 0.7).
        top_p : float, optional
            Nucleus sampling probability (default is 0.95).

        Returns
        -------
        str
            Generated text completion.
        """
        import torch
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt):].strip()

    def generate_stream(
        self,
        prompt: str,
        **kwargs
    ) -> Generator[str, None, None]:
        """
        Generate a text completion in a streaming fashion.

        Parameters
        ----------
        prompt : str
            Input prompt to the model.
        **kwargs
            Additional keyword arguments for model generation.

        Yields
        ------
        str
            Next chunk or line of generated text.
        """
        if self.use_hf:
            output = self.hf_generate(prompt, **kwargs)
            for chunk in output.split("\n"):
                yield chunk
        else:
            # Replace with your actual predibase streaming function
            for chunk in predibase_generate_stream(prompt, **kwargs):  # type: ignore
                yield chunk

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

import os, torch

#base_model_id = "Qwen/Qwen2.5-7B-Instruct"
#base_model_id = "Qwen/Qwen2.5-3B-Instruct"
#base_model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
#base_model_id = "HuggingFaceTB/SmolLM2-135M"
base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"
#base_model_id = "Qwen/Qwen2.5-0.5B-Instruct-GPTQ-Int8"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
llm = LLMHelper(use_hf=True, hf_model_name=base_model_id)
tokenizer = llm.tokenizer

In [ ]:
import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    provider="together",
    api_key=os.environ["HF_TOKEN"],
)

stream = client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ],
    stream=True,
)

for chunk in stream:
    print(chunk.choices[0].delta.content, end="")

## Setting up prompts to play Wordle

In [ ]:
SYSTEM_PROMPT = """
You are playing Wordle, a word-guessing game.

### Game Rules:
- You have **6 tries** to guess a secret **5-letter** word.
- Each guess must be a valid **5-letter English word**.
- After each guess, you will receive feedback indicating how close 
your guess was.

### Feedback Format:
Each letter in your guess will receive one of three symbols:
1. ✓ : The letter is in the word and in the CORRECT position.
2. - : The letter is in the word but in the WRONG position.
3. x : The letter is NOT in the word.

### Example:
Secret Word: BRISK

Guess 1: STORM → Feedback: S(-) T(x) O(x) R(-) M(x)
Guess 2: BRAVE → Feedback: B(✓) R(✓) A(x) V(x) E(x)
Guess 3: BRISK → Feedback: B(✓) R(✓) I(✓) S(✓) K(✓)

### Response Format:
Think through the problem and feedback step by step. Make sure to 
first add your step by step thought process within <think> </think> 
tags. Then, return your guessed word in the following format: 
<guess> guessed-word </guess>.
"""


In [ ]:
from dataclasses import dataclass
from enum import Enum
from typing import List


class LetterFeedback(Enum):
    CORRECT = "✓"
    WRONG_POS = "-"
    WRONG_LETTER = "x"


@dataclass
class GuessWithFeedback:
    guess: str
    feedback: List[LetterFeedback]

    def __repr__(self) -> str:
        """Returns a readable string showing the guess alongside
        its letter-by-letter feedback."""
        feedback_str = " ".join(f"{letter}({fb.value})" for letter, fb in zip(self.guess, self.feedback))
        return f"{self.guess} → Feedback: {feedback_str}"

In [ ]:
def render_user_prompt(past_guesses: List[GuessWithFeedback]) -> str:
    """Creates a user-facing prompt that includes past guesses 
    and their feedback."""
    prompt = "Make a new 5-letter word guess."
    if past_guesses:
        prompt += "\n\nHere is some previous feedback:"
        for i, guess in enumerate(past_guesses):
            prompt += f"\nGuess {i+1}: {guess}"
    return prompt

In [ ]:
def render_prompt(past_guesses: List[GuessWithFeedback], use_local_model: bool = True) -> str:
    """Formats a full chat prompt using a system message, user 
    prompt, and assistant preamble to start the model's 
    step-by-step reasoning."""
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": render_user_prompt(past_guesses)
        },
        {
            "role": "assistant",
            "content": "Let me solve this step by step.\n<think>"
        }
    ]
    if use_local_model:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, continue_final_message=True
        )
    else:
        # Return the raw messages
        return messages

In [ ]:

client = InferenceClient(
    provider="together",
    api_key=os.environ["HF_TOKEN"],
)

stream = client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ],
    stream=True,
)

for chunk in stream:
    print(chunk.choices[0].delta.content, end="")


def generate_stream(prompt: str, model_name: str = "Qwen/Qwen2.5-7B-Instruct") -> str:
    """Streams a model-generated response from a prompt in 
    real-time and prints it as it arrives."""
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        # Produce deterministic responses for evaluation
        temperature=0.0, 
        max_tokens=2048,
        stream=True,
    )
    
    completion = ""
    for chunk in response:
        if chunk.choices[0].text is not None:
            content = chunk.choices[0].text
            print(content, end="", flush=True)
            completion += content
    print()

    return completion

## Play a single turn of Wordle

Start by setting up the prompt with feedback for two prior guesses:

In [ ]:
secret_word = "CRAFT"

past_guesses = [
    GuessWithFeedback(
        "CRANE", [
            LetterFeedback.CORRECT, 
            LetterFeedback.CORRECT, 
            LetterFeedback.CORRECT, 
            LetterFeedback.WRONG_LETTER, 
            LetterFeedback.WRONG_LETTER,
        ]),
    GuessWithFeedback(
        "CRASH", [
            LetterFeedback.CORRECT, 
            LetterFeedback.CORRECT, 
            LetterFeedback.CORRECT, 
            LetterFeedback.WRONG_LETTER, 
            LetterFeedback.WRONG_LETTER,
        ]),
]

prompt = render_prompt(past_guesses, use_local_model=True)
print(prompt)

Prompt the base model and examine its response:

In [ ]:
#base_completion = generate_stream(prompt)
#base_completion = llm.generate_stream(prompt)
#print(list(base_completion))

In [ ]:
# prompt fine-tuned model
#ft_completion = generate_stream(prompt, adapter_id="wordle-dlai/2")

## Playing a game of wordle

Start by setting up a helper function that gets feedback on a guess:

In [ ]:
import re

def get_feedback(guess: str, secret_word: str) -> List[LetterFeedback]:
    valid_letters = set(secret_word)
    feedback = []
    for letter, secret_letter in zip(guess, secret_word):
        if letter == secret_letter:
            feedback.append(LetterFeedback.CORRECT)
        elif letter in valid_letters:
            feedback.append(LetterFeedback.WRONG_POS)
        else:
            feedback.append(LetterFeedback.WRONG_LETTER)
    return feedback

Now create a `next_turn` function that incorporates feedback on a guess into the prompt to the LLM:

In [ ]:
def next_turn(
    past_guesses: List["GuessWithFeedback"], 
    secret_word: str, 
    adapter_id: str = "Qwen/Qwen2.5-7B-Instruct", 
    use_hf: bool = True, 
    llm_helper: Optional["LLMHelper"] = None
) -> None:
    """
    Generate the next guess, update game state, and print feedback.

    Parameters
    ----------
    past_guesses : List[GuessWithFeedback]
        List of previous guesses with feedback.
    secret_word : str
        The word to be guessed.
    adapter_id : str, optional
        Adapter/model identifier (used for Predibase, ignored for HF) (default is "").
    use_hf : bool, optional
        Whether to use Hugging Face LLM (default is False).
    llm_helper : LLMHelper, optional
        Instance of LLMHelper if using Hugging Face.

    Raises
    ------
    RuntimeError
        If a valid guess is not found in the model output.
    """
    prompt = render_prompt(past_guesses, use_local_model=use_hf)

    # Get the model completion
    if use_hf and llm_helper is not None:
        # Assume generate_stream returns a generator, join chunks for full completion
        completion = "".join(chunk for chunk in llm_helper.generate_stream(prompt))
    else:
        completion = generate_stream(prompt, adapter_id)
        if not isinstance(completion, str):
            # If old generate_stream is a generator, join chunks
            completion = "".join(chunk for chunk in completion)

    match = re.search(
        r"<guess>\s*(.*?)\s*</guess>", completion, re.DOTALL
    )
    if not match:
        print("Model did not return a valid guess.")
        print("Here is the full model output:")
        print(completion)
        print("Please try again with a different prompt or model.")
    
    guess = match.group(1).upper()
    print(guess)
    feedback = get_feedback(guess, secret_word)
    past_guesses.append(GuessWithFeedback(guess, feedback))
    print("\n\n")
    print(("-" * 100) + "\n")
    for past_guess in past_guesses:
        print(past_guess)
    
    if guess == secret_word:
        print("🎉 SUCCESS 🎉")
    elif len(past_guesses) >= 6:
        print("❌ better luck next time... ❌")

Try playing with the base model:

In [ ]:
secret_word = "BRICK"
past_guesses = []
adapter_id = ""
for _ in range(2):
    next_turn(past_guesses, secret_word, use_hf=False, llm_helper=llm)

Now try with the finetuned model:

In [ ]:
secret_word = "BRICK"
past_guesses = []
adapter_id = "wordle-dlai/2"

In [ ]:
#next_turn(past_guesses, secret_word, adapter_id)

## Try for yourself!

Try different secret words above and see how the model responds. 